In [2]:
#Task 4.1
import sqlite3

conn = sqlite3.connect("company.db") #Connect to db for holiday company

conn.execute("DROP TABLE IF EXISTS Villa")
conn.execute("DROP TABLE IF EXISTS Bookings")


conn.execute("""CREATE TABLE Villa(
                villaID INTEGER PRIMARY KEY,
                name TEXT,
                country TEXT,
                cost INTEGER
                )""")

conn.execute("""CREATE TABLE Bookings(
                bookingID INTEGER PRIMARY KEY,
                customerID INTEGER,
                villaID INTEGER,
                start_date TEXT,
                num_days TEXT,
                FOREIGN KEY (villaID) REFERENCES Villa
                )""")
    
conn.commit()
conn.close()

In [3]:
#Task 4.2
#helper read function
def read(fileName):
    data = []
    
    file = open(fileName, 'r')
    for line in file:
        line = line.strip().split(',')
        data.append(line)
    
    file.close()
    return data

#Reading
villa = read("villas.txt")
cust = read("customerBookings.txt")

#sql
conn = sqlite3.connect("company.db")

for line in villa:
    conn.execute("INSERT INTO Villa VALUES (?, ?, ?, ?)",
                (line[0], line[1], line[2], line[3]))
    
for line in cust:
    conn.execute("INSERT INTO Bookings VALUES (?, ?, ?, ?, ?)",
                (line[0], line[1], line[2], line[3], line[4]))


conn.commit()
conn.close()

In [4]:
#task 4.3
conn = sqlite3.connect("company.db")

#Tables
conn.execute("DROP TABLE IF EXISTS Villa_Booking")

conn.execute("""CREATE TABLE Villa_Booking(
                villaID INTEGER,
                date TEXT,
                PRIMARY KEY (villaID, date),
                FOREIGN KEY (villaID) REFERENCES Villa
                )""")

#Get data
data = conn.execute("""select Villa.villaID, Bookings.start_date, 
                Bookings.num_days
                from Villa, Bookings
                where Bookings.villaID = Villa.villaID
                order by Villa.villaID asc""").fetchall()

#dictionary of months
months = {
    "Jan" : 31,
    "Feb" : 28,
    "Mar" : 31,
    "Apr" : 30,
    "May" : 31,
    "Jun" : 30,
    "Jul" : 31,
    "Aug" : 31,
    "Sep" : 30,
    "Oct" : 31,
    "Nov" : 30,
    "Dec" : 31
}
next_months = {
    "Jan" : "Feb",
    "Feb" : "Mar",
    "Mar" : "Apr",
    "Apr" : "May",
    "May" : "Jun",
    "Jun" : "Jul",
    "Jul" : "Aug",
    "Aug" : "Oct",
    "Sep" : "Sep",
    "Oct" : "Nov",
    "Nov" : "Dec",
    "Dec" : "Jan"
}

for line in data:
    villa = line[0]
    day, month = line[1].split("-")
    day = int(day)
    num = line[2]
    
    for i in range(int(num)):
        date = str(day).zfill(2) + "-" + month
        #print(villa, date)
        conn.execute("INSERT INTO Villa_Booking VALUES (?, ?)",
                    (villa, date))
        day += 1
        
        if day > int(months[month]):#If we cross to next month
            day = 1
            month = next_months[month]
            #print("Change month")


conn.commit()
conn.close()

In [5]:
#Task 4.4
villa = input("Input Villa Name: ")
month = input("Input Month: ")
day = int(input("Input date: "))
num = int(input("Number of days: "))


#Sql
conn = sqlite3.connect("company.db")

#Return if date is booked
sql = """select Villa.name, Villa_Booking.date
        from Villa, Villa_Booking
        where Villa_Booking.villaID = villa.villaID
        and Villa.name = ?
        and Villa_Booking.date = ?"""

avail = []
not_avail = []

#Months
months = {
    "Jan" : 31,
    "Feb" : 28,
    "Mar" : 31,
    "Apr" : 30,
    "May" : 31,
    "Jun" : 30,
    "Jul" : 31,
    "Aug" : 31,
    "Sep" : 30,
    "Oct" : 31,
    "Nov" : 30,
    "Dec" : 31
}
next_months = {
    "Jan" : "Feb",
    "Feb" : "Mar",
    "Mar" : "Apr",
    "Apr" : "May",
    "May" : "Jun",
    "Jun" : "Jul",
    "Jul" : "Aug",
    "Aug" : "Oct",
    "Sep" : "Sep",
    "Oct" : "Nov",
    "Nov" : "Dec",
    "Dec" : "Jan"
}

#Looping logic
for i in range(num):
    date = str(day).zfill(2) + "-" + month
    booked = conn.execute(sql, (villa, date)).fetchall() 
    
    #dates...
    day += 1
    if day > int(months[month]):
        day = 1
        month = next_months[month]

    
    if len(booked) == 0:
        avail.append(date)
    else:
        not_avail.append(date)

print("Villa is available during: ", avail)
print("Villa is not available during: ", not_avail)
    
conn.close()

Input Villa Name: Dolphin
Input Month: Apr
Input date: 8
Number of days: 4
Villa is available during:  ['08-Apr', '09-Apr']
Villa is not available during:  ['10-Apr', '11-Apr']
